# ⚡ Lab 2: PySpark ETL & Transformation Pipeline

**Welcome back, Kevin!** (Lab Instructor)

## Background

In Lab 1, you profiled DVLA Ghana's legacy data and identified critical quality issues: 
duplicates, missing IDs, mixed date formats, orphan payments, and revenue leakage. Now it's 
time to **fix** these issues using a scalable ETL (Extract, Transform, Load) pipeline.

## What is Apache Spark?

Apache Spark is a **distributed data processing engine** designed to handle massive datasets 
across clusters of machines. Even when running locally (as we do in this lab), Spark provides:

- **Lazy evaluation**: Spark builds a plan (DAG — Directed Acyclic Graph) of transformations 
  but doesn't execute them until you request an action (like `.count()` or `.show()`). This 
  allows Spark to optimize the execution plan.
- **In-memory processing**: Data is kept in RAM between transformations, avoiding slow disk I/O.
- **Fault tolerance**: If a computation fails, Spark can recompute it from the original data.

## Memory Configuration

> ⚠️ **Important**: We limit Spark's driver memory to **2GB** (`spark.driver.memory = "2g"`) 
> because multiple students are running Spark sessions concurrently on this server. Using more 
> memory could cause an Out-of-Memory (OOM) kernel panic that crashes the entire server.

## What You Will Build

| Step | Transformation | Purpose |
|---|---|---|
| 1 | Deduplication | Remove exact duplicate records |
| 2 | Null handling | Flag unverified vehicle owners |
| 3 | Date standardization | Unify to YYYY-MM-DD format |
| 4 | Broadcast join | Map offices to 2026 Zonal Area Codes |
| 5 | Window function | Extract latest payment per vehicle |
| 6 | Revenue flagging | Tag anomalous transactions |
| 7 | Export | Save as Parquet + flat CSV for Power BI |

---

## Step 1: Import Libraries and Create a SparkSession

**📋 What you will do:**
Import the PySpark libraries and create a local SparkSession — the entry point to all 
Spark functionality.

**🔧 Code to type:**
```python
# Import PySpark modules
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType
import os

# Create a local SparkSession with a 2GB memory cap
spark = SparkSession.builder \
    .appName('DVLA_ETL_Kevin') \
    .master('local[*]') \
    .config('spark.driver.memory', '2g') \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.sql.legacy.timeParserPolicy', 'LEGACY') \
    .config('spark.ui.showConsoleProgress', 'false') \
    .getOrCreate()

# Reduce Spark's verbose logging
spark.sparkContext.setLogLevel('WARN')

print(f'Spark version: {spark.version}')
print(f'App name: {spark.sparkContext.appName}')
print('✅ SparkSession created successfully!')
```

**📖 Line-by-line explanation:**
- `SparkSession.builder`: Starts the builder pattern to configure Spark.
- `.appName(...)`: Names your Spark application — visible in the Spark UI for monitoring.
- `.master('local[*]')`: Runs Spark locally using all available CPU cores (`*`). In a 
  production cluster, this would be `yarn` or `mesos`.
- `.config('spark.driver.memory', '2g')`: Limits the JVM heap to 2GB. The driver is the 
  process that coordinates your Spark program.
- `.config('spark.sql.shuffle.partitions', '8')`: Reduces the default 200 shuffle partitions 
  to 8, which is more appropriate for our dataset size (~50K rows).
- `.config('spark.sql.legacy.timeParserPolicy', 'LEGACY')`: Reverts the datetime parsing 
  behavior to legacy mode, which safely returns NULL on pattern mismatches instead of raising exceptions.
- `.getOrCreate()`: Creates a new session or reuses an existing one.

**⚠️ Notes on Common Startup Warnings:**
When running this cell, you will see a few warning lines printed by Java in the console output. 
**These are completely normal, harmless, and expected in any local development environment:**
- `WARNING: Using incubator modules: jdk.incubator.vector`: Java 17+ prints this when Spark 
  activates CPU vector math optimizations for faster calculations.
- `WARN NativeCodeLoader: Unable to load native-hadoop library...`: Occurs because the system 
  uses Spark's built-in Java file reader instead of Hadoop's compiled native C libraries. This has 
  zero impact on local file processing.
- `Using Spark's default log4j profile...`: General setup info logged before your custom 
  log level (`WARN`) takes effect.

**💡 Why this matters for DVLA:**
In a production environment, DVLA could run this same code on a Spark cluster (e.g., AWS EMR 
or Azure HDInsight) to process millions of records. The code structure remains identical — only 
the `.master()` configuration changes.

In [ ]:
# Step 1: Import Libraries and Create SparkSession
# Type your code below


## Step 2: Load the Raw CSV Datasets

**📋 What you will do:**
Read both CSV files from the shared `raw_data` directory into Spark DataFrames.

**🔧 Code to type:**
```python
# Load the vehicle registry
registry_df = spark.read.csv(
    '../raw_data/legacy_vehicle_registry.csv',
    header=True,
    inferSchema=True
)

# Load the payment transactions
payments_df = spark.read.csv(
    '../raw_data/payment_transaction_log.csv',
    header=True,
    inferSchema=True
)

print(f'Registry rows: {registry_df.count():,}')
print(f'Payment rows:  {payments_df.count():,}')
print()
registry_df.printSchema()
```

**📖 Line-by-line explanation:**
- `spark.read.csv(path, header=True, inferSchema=True)`: Reads a CSV file into a Spark 
  DataFrame. `header=True` tells Spark the first row contains column names. `inferSchema=True` 
  asks Spark to automatically detect data types (string, integer, double, etc.).
- `.count()`: An **action** that triggers actual computation. Spark reads the entire file and 
  counts the rows. This is when lazy evaluation ends and real processing begins.
- `.printSchema()`: Displays the column names, data types, and nullability.

**💡 Why this matters for DVLA:**
Note that `Registration_Date` will be inferred as `string` (not date) because of the mixed 
formats. Spark cannot auto-detect dates when multiple formats coexist — we'll fix this in Step 5.

In [ ]:
# Step 2: Load Raw CSV Datasets
# Type your code below


## Step 3: Remove Duplicate Records

**📋 What you will do:**
Remove exact duplicate rows from both datasets using Spark's `dropDuplicates()` method.

**🔧 Code to type:**
```python
# Count before deduplication
reg_before = registry_df.count()
pay_before = payments_df.count()

# Remove exact duplicate rows
registry_df = registry_df.dropDuplicates()
payments_df = payments_df.dropDuplicates()

# Count after deduplication
reg_after = registry_df.count()
pay_after = payments_df.count()

print(f'Registry: {reg_before:,} → {reg_after:,} (removed {reg_before - reg_after:,} duplicates)')
print(f'Payments: {pay_before:,} → {pay_after:,} (removed {pay_before - pay_after:,} duplicates)')
```

**📖 Line-by-line explanation:**
- `dropDuplicates()`: Removes rows that are identical across ALL columns. Spark internally 
  performs a **shuffle** — redistributing data across partitions — to compare rows. This is 
  one of the most expensive operations in Spark.
- We count before and after to quantify the cleanup impact. This metric goes into our 
  dashboard's "Data Cleansing & Quality Audit" view.

**💡 Why this matters for DVLA:**
Deduplication is the first step in any data cleanup. If we skip this, all downstream counts 
(vehicle fleet size, revenue totals) will be inflated by ~5%.

In [ ]:
# Step 3: Remove Duplicate Records
# Type your code below


## Step 4: Handle Missing National ID Numbers

**📋 What you will do:**
Replace null (missing) `National_ID_Number` values with the placeholder `'UNVERIFIED_HOLDER'` 
and add a new column `Identity_Status` that flags each record as `VERIFIED` or `UNVERIFIED_HOLDER`.

**🔧 Code to type:**
```python
# Add Identity_Status column based on whether National_ID is present
registry_df = registry_df.withColumn(
    'Identity_Status',
    F.when(F.col('National_ID_Number').isNull(), 'UNVERIFIED_HOLDER')
     .otherwise('VERIFIED')
)

# Replace null National_ID_Number with placeholder text
registry_df = registry_df.fillna({'National_ID_Number': 'UNVERIFIED_HOLDER'})

# Verify: count by identity status
registry_df.groupBy('Identity_Status').count().show()
```

**📖 Line-by-line explanation:**
- `F.when(condition, value).otherwise(value)`: PySpark's equivalent of SQL's CASE WHEN. 
  Creates a new column based on conditional logic.
- `F.col('National_ID_Number').isNull()`: Checks if the column value is NULL (missing).
- `.fillna({'National_ID_Number': 'UNVERIFIED_HOLDER'})`: Replaces all NULL values in the 
  specified column with the string 'UNVERIFIED_HOLDER'. We do this AFTER creating the status 
  column so the flag is set correctly.
- `.groupBy('Identity_Status').count()`: Quick verification — shows how many records are 
  VERIFIED vs UNVERIFIED_HOLDER.

**💡 Why this matters for DVLA:**
Rather than deleting records with missing IDs (which would lose vehicle data), we flag them. 
This preserves the data while clearly marking which records need follow-up verification 
through the Ghana Card integration process.

In [ ]:
# Step 4: Handle Missing National IDs
# Type your code below


## Step 5: Standardize Date Formats

**📋 What you will do:**
Convert all `Registration_Date` values to a uniform `YYYY-MM-DD` format. The raw data mixes 
two formats: `YYYY-MM-DD` (modern) and `DD/MM/YYYY` (legacy). We use PySpark's `coalesce()` 
to try parsing with both patterns.

**🔧 Code to type:**
```python
# Try parsing as YYYY-MM-DD first, then as DD/MM/YYYY using try_to_date to return NULL on mismatch
registry_df = registry_df.withColumn(
    'Registration_Date',
    F.coalesce(
        F.try_to_date(F.col('Registration_Date'), 'yyyy-MM-dd'),
        F.try_to_date(F.col('Registration_Date'), 'dd/MM/yyyy')
    )
)

# Verify: check for any remaining null dates (would indicate unparseable formats)
null_dates = registry_df.filter(F.col('Registration_Date').isNull()).count()
print(f'Unparseable dates remaining: {null_dates}')

# Show sample of standardized dates
registry_df.select('Registration_ID', 'Registration_Date').show(5)
```

**📖 Line-by-line explanation:**
- `F.try_to_date(col, format)`: Attempts to parse a string column into a date using the 
  specified format pattern. Returns NULL if the string doesn't match the pattern, safely avoiding exceptions.
- `F.coalesce(expr1, expr2)`: Returns the first non-NULL expression. So if the date 
  parses as `yyyy-MM-dd`, use that result. If it returns NULL (wrong format), try 
  `dd/MM/yyyy` instead.
- This two-pass approach handles both formats without any data loss.

**💡 Why this matters for DVLA:**
Standardized dates are essential for time-based reporting (e.g., 'How many vehicles were 
registered in Q1 2024?'). Mixed formats make such queries impossible without preprocessing.

In [ ]:
# Step 5: Standardize Date Formats
# Type your code below


## Step 6: Define the 2026 Zonal Area Code Dictionary

**📋 What you will do:**
Create a small lookup table that maps each DVLA regional office to its new 2026 Zonal Area 
Code. This dictionary will be used in a broadcast join in the next step.

**🔧 Code to type:**
```python
# Define the 2026 Zonal Area Code mapping
# Each regional office maps to a zone code and description
zonal_mapping = [
    ('Accra Metro', 'ACC-Z1', 'Central Accra'),
    ('Weija',       'ACC-Z2', 'West Accra'),
    ('Madina',      'ACC-Z3', 'East Accra'),
    ('Tema',        'ACC-Z4', 'Tema Industrial'),
    ('Kumasi',      'ASH-Z1', 'Kumasi Metro'),
    ('Obuasi',      'ASH-Z2', 'Ashanti South'),
    ('Mampong',     'ASH-Z2', 'Ashanti North'),
    ('Tamale',      'NOR-Z1', 'Northern Metro'),
    ('Takoradi',    'WES-Z1', 'Sekondi-Takoradi'),
    ('Tarkwa',      'WES-Z1', 'Western Mining Belt'),
    ('Cape Coast',  'CEN-Z1', 'Central Metro'),
    ('Winneba',     'CEN-Z1', 'Central Coast'),
    ('Koforidua',   'EAS-Z1', 'Eastern Metro'),
    ('Nkawkaw',     'EAS-Z1', 'Eastern Highlands'),
    ('Sunyani',     'BON-Z1', 'Bono Metro'),
    ('Techiman',    'BON-Z1', 'Bono East'),
    ('Bolgatanga',  'UPE-Z1', 'Upper East Metro'),
    ('Wa',          'UPW-Z1', 'Upper West Metro'),
    ('Ho',          'VOL-Z1', 'Volta Metro'),
    ('Aflao',       'VOL-Z1', 'Volta Border'),
]

# Create a Spark DataFrame from the mapping
zonal_schema = StructType([
    StructField('Regional_Office', StringType(), False),
    StructField('Zonal_Area_Code', StringType(), False),
    StructField('Zone_Description', StringType(), False),
])

zonal_df = spark.createDataFrame(zonal_mapping, schema=zonal_schema)
zonal_df.show(truncate=False)
print(f'Zonal mapping entries: {zonal_df.count()}')
```

**📖 Line-by-line explanation:**
- `zonal_mapping`: A Python list of tuples. Each tuple contains three values: the current 
  Regional_Office name, the new Zonal_Area_Code, and a human-readable Zone_Description.
- `StructType([...])`: Explicitly defines the schema instead of relying on inference. This is 
  best practice for small lookup tables where you want guaranteed data types.
- `spark.createDataFrame(data, schema)`: Converts a Python list into a Spark DataFrame.

**💡 Why this matters for DVLA:**
The 2026 Zonal Area Code harmonization is a real policy initiative to reorganize DVLA's 
administrative boundaries. This mapping ensures all vehicle records can be classified under 
the new zonal structure for decentralized reporting.

In [ ]:
# Step 6: Define Zonal Area Code Dictionary
# Type your code below


## Step 7: Broadcast Join — Map Offices to Zonal Codes

**📋 What you will do:**
Join the vehicle registry with the Zonal Area Code dictionary using a **broadcast join**. 
This is a high-performance technique for joining a large table with a small lookup table.

**🔧 Code to type:**
```python
from pyspark.sql.functions import broadcast

# Perform a broadcast join
# The small zonal_df is broadcast (copied) to every executor node
registry_df = registry_df.join(
    broadcast(zonal_df),
    on='Regional_Office',
    how='left'
)

# Verify: check for unmatched offices (should be 0 if mapping is complete)
unmatched = registry_df.filter(F.col('Zonal_Area_Code').isNull()).count()
print(f'Unmatched regional offices: {unmatched}')

# Show sample with zonal codes
registry_df.select('Registration_ID', 'Regional_Office', 'Zonal_Area_Code', 'Zone_Description').show(10)
```

**📖 Line-by-line explanation:**
- `broadcast(zonal_df)`: Tells Spark to send a complete copy of the small DataFrame to every 
  worker node. This avoids an expensive shuffle of the large registry DataFrame.
- `on='Regional_Office'`: The join key — rows are matched where the office names are equal.
- `how='left'`: A left join keeps all registry rows, even if no zonal match is found (the 
  zonal columns would be NULL for unmatched rows).

**⚡ Performance insight:**
Without `broadcast()`, Spark would shuffle both DataFrames across the network — moving ~50,000 
rows. With `broadcast()`, only the 20-row lookup table is copied. This is orders of magnitude 
faster and uses far less memory.

**💡 Why this matters for DVLA:**
Every vehicle record now carries its 2026 Zonal Area Code, enabling decentralized reporting 
without needing to look up the mapping each time.

In [ ]:
# Step 7: Broadcast Join
# Type your code below


## Step 8: Prepare Payment Data and Flag Revenue Anomalies

**📋 What you will do:**
Clean the payment dataset and add a `Revenue_Flag` column that categorizes each transaction 
as CLEAN, ZERO_OR_NEGATIVE, or based on the payment amount.

**🔧 Code to type:**
```python
# Add Revenue_Flag based on payment amount
payments_df = payments_df.withColumn(
    'Revenue_Flag',
    F.when(F.col('Amount_Paid_GHS') <= 0, 'ZERO_OR_NEGATIVE')
     .otherwise('CLEAN')
)

# Verify: count by revenue flag
payments_df.groupBy('Revenue_Flag').count().show()
```

**📖 Line-by-line explanation:**
- `F.when(condition, value).otherwise(value)`: Creates a new column using conditional logic.
- Payments with amount ≤ 0 are flagged as `ZERO_OR_NEGATIVE`. All others are `CLEAN`.
- The `.groupBy().count()` verification shows how many transactions fall into each category.

**💡 Why this matters for DVLA:**
This Revenue_Flag column becomes the basis for the 'Revenue Assurance & Leakage Map' in our 
dashboard. Deputy Director Albert can filter to see only flagged transactions for investigation.

In [ ]:
# Step 8: Prepare Payments and Flag Anomalies
# Type your code below


## Step 9: Extract the Latest Payment Per Vehicle (Window Function)

**📋 What you will do:**
Use a PySpark Window function to find the most recent payment transaction for each vehicle. 
This is essential for tracking active verification status.

**🔧 Code to type:**
```python
# Define a window: partition by Registration_ID, order by timestamp descending
payment_window = Window.partitionBy('Registration_ID').orderBy(F.desc('Payment_Timestamp'))

# Add a row number within each partition (1 = most recent)
payments_ranked = payments_df.withColumn(
    'row_num',
    F.row_number().over(payment_window)
)

# Keep only the latest payment per vehicle (row_num = 1)
latest_payments = payments_ranked.filter(F.col('row_num') == 1).drop('row_num')

# Rename columns to indicate these are the 'latest' values
latest_payments = latest_payments \
    .withColumnRenamed('Transaction_ID', 'Latest_Transaction_ID') \
    .withColumnRenamed('Amount_Paid_GHS', 'Latest_Amount_Paid_GHS') \
    .withColumnRenamed('Payment_Channel', 'Latest_Payment_Channel') \
    .withColumnRenamed('Payment_Timestamp', 'Latest_Payment_Timestamp') \
    .withColumnRenamed('Revenue_Flag', 'Revenue_Flag')

print(f'Unique vehicles with payments: {latest_payments.count():,}')
latest_payments.show(5)
```

**📖 Line-by-line explanation:**
- `Window.partitionBy('Registration_ID')`: Groups rows by vehicle. Each vehicle gets its own 
  'window' of payment records.
- `.orderBy(F.desc('Payment_Timestamp'))`: Within each window, sorts payments from newest to 
  oldest. The most recent payment gets position 1.
- `F.row_number().over(window)`: Assigns sequential integers (1, 2, 3...) within each window.
- `.filter(F.col('row_num') == 1)`: Keeps only the top row (most recent) per vehicle.

**💡 Why this matters for DVLA:**
A vehicle may have multiple payment records (initial registration, annual renewals). We need 
the latest one to determine current compliance status. The window function does this 
efficiently without writing complex subqueries.

In [ ]:
# Step 9: Window Function — Latest Payment Per Vehicle
# Type your code below


## Step 10: Join Registry with Latest Payments

**📋 What you will do:**
Left join the cleaned vehicle registry with the latest payment records. Vehicles with no 
payment will be flagged as `NO_PAYMENT_RECORD`. Orphan payments (those with Registration_IDs 
not in the registry) will be flagged as `ORPHAN_PAYMENT`.

**🔧 Code to type:**
```python
# Left join: keep all vehicles, attach their latest payment
final_df = registry_df.join(
    latest_payments,
    on='Registration_ID',
    how='left'
)

# Update Revenue_Flag for vehicles with no payment record
final_df = final_df.withColumn(
    'Revenue_Flag',
    F.when(F.col('Latest_Transaction_ID').isNull(), 'NO_PAYMENT_RECORD')
     .otherwise(F.col('Revenue_Flag'))
)

# Show the distribution of revenue flags
final_df.groupBy('Revenue_Flag').count().orderBy('count', ascending=False).show()
print(f'Final dataset rows: {final_df.count():,}')
```

**📖 Line-by-line explanation:**
- `how='left'`: All registry vehicles are kept. If a vehicle has no matching payment, the 
  payment columns will be NULL.
- `F.when(F.col('Latest_Transaction_ID').isNull(), 'NO_PAYMENT_RECORD')`: Vehicles with no 
  payment get this flag. This could indicate a vehicle that was registered but never paid 
  fees — a compliance issue.
- `.otherwise(F.col('Revenue_Flag'))`: Preserves the existing flag (CLEAN or ZERO_OR_NEGATIVE) 
  for vehicles that do have a payment record.

**💡 Why this matters for DVLA:**
The final Revenue_Flag provides a complete picture: CLEAN (legitimate), ZERO_OR_NEGATIVE 
(suspicious), NO_PAYMENT_RECORD (compliance gap). This directly feeds the Revenue Assurance 
dashboard view.

In [ ]:
# Step 10: Join Registry with Latest Payments
# Type your code below


## Step 11: Create Output Directory

**📋 What you will do:**
Create the `./output/` directory within your workspace to store the cleaned data files.

**🔧 Code to type:**
```python
# Create output directories
os.makedirs('./output/parquet', exist_ok=True)
print('✅ Output directory created: ./output/')
```

**📖 Line-by-line explanation:**
- `os.makedirs(path, exist_ok=True)`: Creates the directory and all parent directories. 
  `exist_ok=True` prevents an error if the directory already exists.

In [ ]:
# Step 11: Create Output Directory
# Type your code below


## Step 12: Save as Partitioned Parquet Files

**📋 What you will do:**
Save the cleaned dataset as partitioned Parquet files. Parquet is a columnar storage format 
optimized for analytical queries — it compresses data significantly and enables fast reads.

**🔧 Code to type:**
```python
# Write partitioned Parquet files (organized by Zonal Area Code)
final_df.write \
    .mode('overwrite') \
    .partitionBy('Zonal_Area_Code') \
    .parquet('./output/parquet/')

print('✅ Parquet files saved to ./output/parquet/')
```

**📖 Line-by-line explanation:**
- `.write`: Initiates a write operation on the DataFrame.
- `.mode('overwrite')`: Replaces any existing files in the output directory.
- `.partitionBy('Zonal_Area_Code')`: Creates subdirectories for each zone (e.g., 
  `Zonal_Area_Code=ACC-Z1/`, `Zonal_Area_Code=ASH-Z1/`). This enables partition pruning — 
  queries targeting a specific zone only read that folder.
- `.parquet(path)`: Writes in Apache Parquet format.

**💡 Why this matters for DVLA:**
Parquet files are the standard format for big data pipelines. They are 5-10x smaller than 
CSV and can be read directly by Spark, Athena, BigQuery, and other analytics tools.

In [ ]:
# Step 12: Save as Partitioned Parquet
# Type your code below


## Step 13: Export Clean Data as a Flat CSV for Power BI

**📋 What you will do:**
Convert the Spark DataFrame to a Pandas DataFrame and save it as a single flat CSV file. 
This is necessary because Spark's `.write.csv()` creates a directory with multiple part files, 
which Power BI cannot read directly.

**🔧 Code to type:**
```python
# Convert Spark DataFrame to Pandas and save as a single CSV
# This consolidates all distributed data to a single node
final_pandas = final_df.toPandas()
final_pandas.to_csv('./output/powerbi_ready.csv', index=False, encoding='utf-8')

print(f'✅ Saved {len(final_pandas):,} rows to ./output/powerbi_ready.csv')
print(f'Columns: {list(final_pandas.columns)}')
```

**📖 Line-by-line explanation:**
- `.toPandas()`: Collects all data from the distributed Spark DataFrame to a single-node 
  Pandas DataFrame. **Warning**: This works here because our dataset is small (~50K rows). 
  For millions of rows, you would need a different approach.
- `.to_csv(path, index=False, encoding='utf-8')`: Saves as a standard CSV file. `index=False` 
  prevents Pandas from adding a row number column. `encoding='utf-8'` ensures Ghanaian names 
  with special characters are preserved correctly.

**⚠️ Why not use Spark's .write.csv()?**
Spark's `.write.csv()` creates a directory containing multiple files like `part-00000.csv`, 
`part-00001.csv`, plus a `_SUCCESS` marker file. Power BI and Streamlit expect a single file 
at a known path — not a directory of parts.

In [ ]:
# Step 13: Export as Flat CSV
# Type your code below


## Step 14: Validate the Clean Output

**📋 What you will do:**
Run validation checks to confirm the ETL pipeline produced correct results.

**🔧 Code to type:**
```python
import pandas as pd

# Reload the saved CSV and validate
validation_df = pd.read_csv('./output/powerbi_ready.csv', encoding='utf-8')

print('=== VALIDATION REPORT ===')
print(f'Total rows:           {len(validation_df):,}')
print(f'Total columns:        {len(validation_df.columns)}')
print(f'Columns:              {list(validation_df.columns)}')
print()

# Check for expected columns
expected_cols = ['Registration_ID', 'Chassis_Number', 'Owner_Name', 'National_ID_Number',
                 'Registration_Date', 'Vehicle_Make', 'Regional_Office',
                 'Zonal_Area_Code', 'Zone_Description', 'Identity_Status']
missing = [c for c in expected_cols if c not in validation_df.columns]
print(f'Missing expected columns: {missing if missing else "None ✅"}')
print()

# Check data completeness
if 'Zonal_Area_Code' in validation_df.columns:
    null_zones = validation_df['Zonal_Area_Code'].isna().sum()
    print(f'Null Zonal_Area_Code: {null_zones} (should be 0)')

if 'Identity_Status' in validation_df.columns:
    print(f'Identity breakdown:')
    print(validation_df['Identity_Status'].value_counts().to_string())
print()

if 'Revenue_Flag' in validation_df.columns:
    print(f'Revenue flag breakdown:')
    print(validation_df['Revenue_Flag'].value_counts().to_string())

print()
print('✅ Validation complete!')
```

**📖 Explanation:**
This validation step confirms that:
- The output file exists and is readable
- All expected columns are present
- Zonal Area Codes were successfully assigned (no nulls)
- Identity and Revenue flags are populated

In [ ]:
# Step 14: Validate the Clean Output
# Type your code below


## Step 15: Stop the SparkSession

**📋 What you will do:**
Shut down the SparkSession to release JVM memory and resources.

**🔧 Code to type:**
```python
# Stop the SparkSession to free memory
spark.stop()
print('✅ SparkSession stopped. JVM resources released.')
```

**📖 Explanation:**
- `spark.stop()`: Shuts down the Spark driver process and frees the 2GB of JVM memory. 
  This is **critical** on a shared server — if you don't stop Spark, the memory remains 
  allocated and other students may not be able to start their sessions.

> ⚠️ **Always run this cell when you finish Lab 2!**

In [ ]:
# Step 15: Stop the SparkSession
# Type your code below


## 📊 Next Steps: Downloading Data & Building a Power BI Star Schema

Congratulations, Kevin! You have completed the ETL pipeline. Your clean data is now 
saved at `./output/powerbi_ready.csv`.

### Step A: Download Your Clean CSV File

1. In the **JupyterLab sidebar** (left panel), click the 📁 file browser icon.
2. Navigate to your workspace folder → `output/`.
3. Right-click on `powerbi_ready.csv`.
4. Select **"Download"** from the context menu.
5. The file will be saved to your **Windows 11 Downloads folder**.

### Step B: Import into Power BI Desktop

1. Open **Power BI Desktop** on your Windows 11 machine.
2. Click **Home → Get Data → Text/CSV**.
3. Browse to your Downloads folder and select `powerbi_ready.csv`.
4. In the preview dialog, verify the column types look correct, then click **Load**.

### Step C: Build a Star Schema Data Model

A **Star Schema** is a database design pattern optimized for analytical reporting. It consists of:
- **Fact Table**: The main data table containing measures (amounts, counts).
- **Dimension Tables**: Small lookup tables for filtering and grouping.

**To create the Star Schema:**

1. After importing `powerbi_ready.csv`, go to the **Model view** (left sidebar icon).
2. Your imported data becomes the **Fact Table** (`powerbi_ready`).
3. Create a **Zonal Lookup Dimension Table**:
   - Click **Home → Enter Data**.
   - Create a table named `Dim_Zonal_Lookup` with columns: `Zonal_Area_Code`, `Zone_Description`, `Region`.
   - Fill in the 10 unique zonal codes from the mapping (ACC-Z1, ACC-Z2, etc.).
4. **Create the Relationship**:
   - In the Model view, drag `Zonal_Area_Code` from the Fact Table to `Zonal_Area_Code` in `Dim_Zonal_Lookup`.
   - Set cardinality to **Many-to-One (N:1)** — many vehicles belong to one zone.
   - Set cross-filter direction to **Single**.

### Star Schema Diagram

```
                    ┌──────────────────────┐
                    │   Dim_Zonal_Lookup   │
                    │──────────────────────│
                    │ Zonal_Area_Code (PK) │
                    │ Zone_Description     │
                    │ Region               │
                    └─────────┬────────────┘
                              │  1:N
                    ┌─────────┴────────────┐
                    │   powerbi_ready      │
                    │   (Fact Table)        │
                    │──────────────────────│
                    │ Registration_ID      │
                    │ Zonal_Area_Code (FK) │
                    │ Latest_Amount_Paid   │
                    │ Revenue_Flag         │
                    │ Identity_Status      │
                    │ ...                  │
                    └──────────────────────┘
```

---
*Lab 2 Complete — Well done, Kevin! 🎉*

*You can now view your results on the Streamlit dashboard or continue to build 
your Power BI report.*